In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [3]:
query = """
SELECT *
FROM customer_features
"""

customer_features = pd.read_sql(
    query,
    engine
)

In [4]:
customer_features.shape
customer_features.head()
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   total_orders            96096 non-null  int64         
 2   total_revenue           96096 non-null  float64       
 3   avg_order_value         96096 non-null  float64       
 4   avg_review_score        96096 non-null  float64       
 5   total_freight           96096 non-null  float64       
 6   avg_installments        96096 non-null  float64       
 7   first_purchase          96096 non-null  datetime64[us]
 8   last_purchase           96096 non-null  datetime64[us]
 9   customer_lifetime_days  96096 non-null  int64         
 10  recency_days            96096 non-null  int64         
 11  repeat_customer         96096 non-null  int64         
 12  customer_tenure_months  96096 non-null  float64       
 1

In [5]:
customer_features.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_revenue',
 'avg_order_value',
 'avg_review_score',
 'total_freight',
 'avg_installments',
 'first_purchase',
 'last_purchase',
 'customer_lifetime_days',
 'recency_days',
 'repeat_customer',
 'customer_tenure_months',
 'revenue_per_day',
 'high_value_customer',
 'freight_percentage',
 'review_category',
 'recency_group',
 'R_score',
 'F_score',
 'M_score',
 'RFM_score',
 'customer_value_score']

In [6]:
# target variable
y = customer_features["repeat_customer"]

In [7]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

repeat_customer
0    93099
1     2997
Name: count, dtype: int64

In [8]:
# checking for repeat customers in percentage
customer_features["repeat_customer"].value_counts(normalize=True) * 100

repeat_customer
0    96.881244
1     3.118756
Name: proportion, dtype: float64

In [9]:
# checking for missng values
customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [10]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

customer_features["repeat_customer"].value_counts(normalize=True) * 100

customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [11]:
# 
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [12]:
customer_features["review_category"] = (
    customer_features["review_category"]
    .fillna("Unknown")
)

customer_features["recency_group"] = (
    customer_features["recency_group"]
    .fillna("Unknown")
)

In [13]:
customer_features.isnull().sum().sum()

np.int64(0)

In [14]:
y = customer_features["repeat_customer"]

In [15]:
leakage_columns = [
    "total_orders",
    "F_score",
    "RFM_score",
    "customer_value_score"
]

In [16]:
X = customer_features.drop(
    columns=[
        "customer_unique_id",
        "repeat_customer",
        "first_purchase",
        "last_purchase"
    ] + leakage_columns
)

In [17]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [18]:
import numpy as np

X.replace(
    [np.inf, -np.inf],
    0,
    inplace=True
)

,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,customer_lifetime_days,recency_days,customer_tenure_months,revenue_per_day,high_value_customer,...,recency_group_Very Recent,recency_group_Warm,R_score_2,R_score_3,R_score_4,R_score_5,M_score_2,M_score_3,M_score_4,M_score_5
0,141.90,141.90,5.0,12.00,8.0,0,160,0.0,141.90,0,...,False,True,False,False,True,False,False,False,True,False
1,27.19,27.19,4.0,8.29,1.0,0,163,0.0,27.19,0,...,False,True,False,False,True,False,False,False,False,False
2,86.22,86.22,3.0,17.22,8.0,0,585,0.0,86.22,0,...,False,False,False,False,False,False,True,False,False,False
3,43.62,43.62,4.0,17.63,4.0,0,369,0.0,43.62,0,...,False,False,True,False,False,False,False,False,False,False
4,196.89,196.89,5.0,16.89,6.0,0,336,0.0,196.89,0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,4134.84,2067.42,5.0,497.42,10.0,0,495,0.0,4134.84,1,...,False,False,False,False,False,False,False,False,False,True
96092,84.58,84.58,4.0,19.69,1.0,0,310,0.0,84.58,0,...,False,False,False,True,False,False,True,False,False,False
96093,112.46,112.46,5.0,22.56,1.0,0,617,0.0,112.46,0,...,False,False,False,False,False,False,False,True,False,False
96094,133.69,133.69,5.0,18.69,5.0,0,168,0.0,133.69,0,...,False,True,False,False,True,False,False,True,False,False


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
scale_pos_weight = (
    y_train.value_counts()[0]
    /
    y_train.value_counts()[1]
)

print(scale_pos_weight)

31.058381984987488


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
rf_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap:

In [23]:
from sklearn.metrics import classification_report

y_pred = rf_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     18621
           1       1.00      0.73      0.84       599

    accuracy                           0.99     19220
   macro avg       1.00      0.86      0.92     19220
weighted avg       0.99      0.99      0.99     19220



In [24]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.99      1.00      1.00     18621\n           1       1.00      0.73      0.84       599\n\n    accuracy                           0.99     19220\n   macro avg       1.00      0.86      0.92     19220\nweighted avg       0.99      0.99      0.99     19220\n'

In [25]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[18621     0]
 [  162   437]]


In [26]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.9915712799167534
Precision: 1.0
Recall   : 0.7295492487479132
F1 Score : 0.8436293436293436


In [27]:
train_pred =rf_model.predict(X_train)

print(
    "Train Accuracy:",
    accuracy_score(y_train, train_pred)
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

Train Accuracy: 0.9907253238982258
Test Accuracy: 0.9915712799167534


In [28]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

importance.sort_values(
    by="importance",
    ascending=False
).head(20)

,feature,importance
7,customer_tenure_months,0.350977
5,customer_lifetime_days,0.350424
8,revenue_per_day,0.220440
1,avg_order_value,0.025707
3,total_freight,0.019141
0,total_revenue,0.011756
10,freight_percentage,0.005012
9,high_value_customer,0.004437
2,avg_review_score,0.002473
26,M_score_5,0.002127


In [29]:
prob = rf_model.predict_proba(X_test)

prob[:5]

array([[0.99756146, 0.00243854],
       [0.99834034, 0.00165966],
       [0.99314579, 0.00685421],
       [0.97895745, 0.02104255],
       [0.99829429, 0.00170571]])

In [30]:
from sklearn.metrics import roc_auc_score

prob = rf_model.predict_proba(X_test)[:,1]

roc_auc_score(
    y_test,
    prob
)

0.9891293501628432

In [31]:
model_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test,y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, prob),
})

model_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.991571,1.0,0.729549,0.843629,0.989129


In [32]:
# STORING ACCURACY, PRECISION, RECALL, F1 SCORE AND ROC AUC OF RANDOM FOREST IN POSTGRESQL DATABASE
from datetime import datetime

rf_results = pd.DataFrame({
    "model_name": ["RANDOM FOREST"],
    "accuracy": [accuracy_score(y_test, y_pred)],
    "precision": [precision_score(y_test, y_pred)],
    "recall": [recall_score(y_test, y_pred)],
    "f1_score": [f1_score(y_test, y_pred)],
    "roc_auc": [roc_auc_score(y_test, prob)],
    "run_date": [datetime.now()]
})

from sqlalchemy import text

model_name = "RANDOM FOREST"

with engine.begin() as conn:
    conn.execute(
        text("DELETE FROM model_metrics WHERE model_name = :model"),
        {"model": model_name}
    )

rf_results.to_sql(
    "model_metrics",
    con=engine,
    if_exists="append",
    index=False
)

1

In [33]:
# storing file into trained_models/purchase_prediction/random_forest.pkl
import joblib

import joblib
from pathlib import Path

BASE_DIR = Path.cwd()

MODEL_DIR = BASE_DIR / "Trained_Models" / "purchase_prediction"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    rf_model,
    MODEL_DIR / "random_forest.pkl"
)

['C:\\Users\\vansh\\OneDrive\\Desktop\\sureTrust\\Trained_Models\\purchase_prediction\\random_forest.pkl']